# 特征工程文档宏观总结
核心目标：将推荐召回的候选结果，转化为排序模型可训练的**监督学习数据集**，完成排序任务的全流程特征工程。

## 1. 问题转换
将推荐召回的{user_id: [可能点击的文章列表]}，转化为排序任务的监督学习问题，目标是预测用户是否点击候选文章。

## 2. 基础数据处理
- 内存优化：用`reduce_mem`函数压缩数据集，降低内存占用
- 数据集划分：拆分训练/验证/测试集，提取用户「历史点击」和「最后一次点击」（作为标签依据）
- 数据读取：加载文章信息、各类Embedding（内容Embedding、Word2Vec、YouTubeDNN生成的用户/文章Embedding）

## 3. 召回数据标准化
1. 格式转换：将召回字典转为结构化DataFrame（用户、候选文章、得分）
2. 样本打标：用户最后点击的候选为正样本（1），未点击为负样本（0）
3. 负采样：解决正负样本不均衡问题，减少特征计算压力，保证所有用户/文章均在采样后数据中

## 4. 核心特征构建（三大类）
### （1）用户-候选历史行为特征
- 候选文章与用户最后N次点击的相似度（Embedding内积）及统计特征（最大/最小/均值/和）
- 候选与用户最后N次点击的时间差、字数差特征

### （2）用户画像特征
- 活跃度：基于点击次数、点击时间间隔衡量
- 设备习惯：用户最常用的设备相关信息（众数）
- 时间偏好：点击时间、文章创建时间的统计均值
- 主题爱好：用户历史点击文章的主题列表
- 字数偏好：用户历史点击文章的字数均值

### （3）文章画像特征
- 热度：基于被点击次数、点击时间间隔衡量
- 自身属性：文章类型（category_id）、时效（created_at_ts）、字数（words_count）

### （4）交叉特征
- 候选文章主题是否匹配用户历史爱好（0/1特征）

## 5. 特征融合与落地
- 拼接所有特征，删除冗余列（如文章ID、主题列表）
- 保存训练/验证/测试特征文件，用于后续排序模型（如LightGBM）训练

### 简单总结
将「召回的候选文章」加工成「带标签、全特征」的标准数据集，为排序模型提供可训练的数据基础。

In [1]:
!pip install lightgbm gensim scikit-learn
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import gc, os
import logging
import time
import lightgbm as lgb
from gensim.models import Word2Vec
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 节省内存的一个函数
# 减少内存
def reduce_mem(df):
    starttime = time.time()
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.isnull(c_min) or pd.isnull(c_max):
                continue
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    print('-- Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction),time spend:{:2.2f} min'.format(end_mem,
                                                                                                           100*(start_mem-end_mem)/start_mem,
                                                                                                           (time.time()-starttime)/60))
    return df

data_path = './data/'
save_path = './temp_results/'

In [3]:
# all_click_df指的是训练集
# sample_user_nums 采样作为验证集的用户数量
def trn_val_split(all_click_df, sample_user_nums):
    all_click = all_click_df
    all_user_ids = all_click.user_id.unique()
    
    # replace=True表示可以重复抽样，反之不可以
    sample_user_ids = np.random.choice(all_user_ids, size=sample_user_nums, replace=False) 
    
    click_val = all_click[all_click['user_id'].isin(sample_user_ids)]
    click_trn = all_click[~all_click['user_id'].isin(sample_user_ids)]
    
    # 将验证集中的最后一次点击给抽取出来作为答案
    click_val = click_val.sort_values(['user_id', 'click_timestamp'])
    val_ans = click_val.groupby('user_id').tail(1)
    
    click_val = click_val.groupby('user_id').apply(lambda x: x[:-1]).reset_index(drop=True)
    
    # 去除val_ans中某些用户只有一个点击数据的情况，如果该用户只有一个点击数据，又被分到ans中，
    # 那么训练集中就没有这个用户的点击数据，出现用户冷启动问题，给自己模型验证带来麻烦
    val_ans = val_ans[val_ans.user_id.isin(click_val.user_id.unique())] # 保证答案中出现的用户再验证集中还有
    click_val = click_val[click_val.user_id.isin(val_ans.user_id.unique())]
    
    return click_trn, click_val, val_ans

# 特征工程核心思路总结
## 一、可直接复用的原始特征
从给定数据中，可直接取用的基础特征：
1. **文章自身特征**
    - 文章类别 `category_id`
    - 发布时间 `created_at_ts`（体现时效性）
    - 文章字数 `words_count`（反映用户阅读偏好）
2. **文章内容特征**
    内容 Embedding、Word2Vec、YouTubeDNN 等向量特征
3. **用户基础特征**
    用户设备相关信息

这些特征后续可通过 `article_id` / `user_id` 直接拼接加入数据集。

## 二、构造监督学习数据集
1. **数据格式转换**
    把召回结果 `{user_id: [候选文章列表]}` 展开为结构化样本：
    `(用户ID, 候选文章ID)` 对，作为排序模型的基础样本行。
2. **核心目标**
    预测用户是否会点击该候选文章，将推荐问题转化为**有监督的二分类任务**。

## 三、核心特征构造思路
围绕**用户历史点击行为**构建关键特征，重点关联用户**最后几次点击**（贴合新闻时效性）：
1. 候选文章与用户最后几次点击文章的 **Embedding 相似度（内积）**
2. 上述相似度的统计特征（均值、最大/最小等，平滑波动）
3. 候选与历史点击文章的**字数差**（刻画阅读长度偏好）
4. 候选与历史点击文章的**发布时间差**（刻画时效性偏好）
5. 若使用 YouTube 召回，额外增加**用户与候选文章的相似度特征**

## 四、整体实现流程
1. 从日志数据中拆分：用户**最后一次点击**（用于打标签）+ 用户**历史点击**
2. 结合召回列表、文章信息、Embedding 向量，完成上述行为特征构建
3. 为样本制作标签，最终形成可直接用于模型训练的监督学习数据集

In [4]:
# 获取当前数据的历史点击和最后一次点击
def get_hist_and_last_click(all_click):
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df

# 读取训练、验证及测试集
def get_trn_val_tst_data(data_path, offline=True):
    if offline:
        click_trn_data = pd.read_csv(data_path+'train_click_log.csv')  # 训练集用户点击日志
        click_trn_data = reduce_mem(click_trn_data)
        click_trn, click_val, val_ans = trn_val_split(all_click_df, sample_user_nums)
    else:
        click_trn = pd.read_csv(data_path+'train_click_log.csv')
        click_trn = reduce_mem(click_trn)
        click_val = None
        val_ans = None
    
    click_tst = pd.read_csv(data_path+'testA_click_log.csv')
    
    return click_trn, click_val, click_tst, val_ans

# 读取召回列表
# 返回多路召回列表或者单路召回
def get_recall_list(save_path, single_recall_model=None, multi_recall=False):
    if multi_recall:
        return pickle.load(open(save_path + 'final_recall_items_dict.pkl', 'rb'))
    
    if single_recall_model == 'i2i_itemcf':
        return pickle.load(open(save_path + 'itemcf_recall_dict.pkl', 'rb'))
    elif single_recall_model == 'i2i_emb_itemcf':
        return pickle.load(open(save_path + 'itemcf_emb_dict.pkl', 'rb'))
    elif single_recall_model == 'user_cf':
        return pickle.load(open(save_path + 'youtubednn_usercf_dict.pkl', 'rb'))
    elif single_recall_model == 'youtubednn':
        return pickle.load(open(save_path + 'youtube_u2i_dict.pkl', 'rb'))


# 读取Embedding & Word2Vec训练 通俗总结
## 核心目的
给**文章、用户**做**数字化画像（Embedding向量）**
- 模型看不懂文章ID、文字，只能看懂**数字**
- 把文章/用户转成一串固定长度的数字，就能轻松计算**相似度**（比如：这篇新文章和用户喜欢的文章像不像）

---

## 一、Word2Vec 到底是啥？（推荐场景版）
### 课本定义（简单记）
无监督学习算法，通过**上下文**生成词向量；两个模式：
- CBOW：用周围词猜中心词
- Skip-Gram：用中心词猜周围词（推荐里常用这个）

### 🔥 推荐里的魔改用法（重点！）
我们**不拿它处理文字**，而是用来生成**文章向量**：
1. 把**用户的点击历史** → 当成一个「句子」
   例：用户先点文章1 → 文章5 → 文章9 → 句子：`[1,5,9]`
2. 把**文章ID** → 当成句子里的「单词」
3. 用Word2Vec训练：**经常被同一个用户点击的文章，生成的向量会非常相似**

---

## 二、核心代码1：训练文章的Word2Vec向量
函数 `trian_item_word2vec`（笔误，应该是train）**专门生成文章的点击行为向量**
### 逐行大白话：
1. 按点击时间排序，保证用户点击顺序是对的
2. 把每个用户点过的所有文章，拼成一个列表（句子）
3. 用这些「点击句子」训练Word2Vec
4. 生成一个字典：`{文章ID: 对应的数字向量}`
5. 保存到文件，下次直接用，不用重复训练

### 关键参数（不用背，知道意思就行）
- `sg=1`：用效果更好的 Skip-Gram 模型
- `window=5`：看用户点击序列前后5篇文章的关联
- `size=16`：每篇文章对应16个数字的向量
- `min_count=1`：所有文章都参与训练，不剔除冷门文章

---

## 三、核心代码2：读取所有Embedding
函数 `get_embedding` **统一加载所有预训练好的向量**
我们一共用4种向量，这个函数一次性全读出来：
1. **文章内容向量**：文章本身的文字内容生成（自带的）
2. **文章Word2Vec向量**：刚才训练的（基于用户点击行为）
3. **文章YouTube向量**：召回模型生成的
4. **用户YouTube向量**：召回模型生成的

### 逻辑：
- 检查文件是否存在 → 存在直接读取
- 不存在 → 就训练Word2Vec向量
- 最后把4个向量一起返回，给后面**特征工程计算相似度**用

---

## 四、这部分的最终作用（一句话）
把**文章、用户**转成模型能识别的**数字向量**，为后面计算「候选文章和用户历史点击的相似度」提供核心数据，是特征工程的关键原料。

In [5]:
# 把用户的点击历史训练成文章的行为向量，生成一个 {文章ID: 向量} 字典，方便后面算相似度
def trian_item_word2vec(click_df, embed_size=64, save_name='item_w2v_emb.pkl', split_char=' '):
    click_df = click_df.sort_values('click_timestamp')
    # 只有转换成字符串才可以进行训练
    click_df['click_article_id'] = click_df['click_article_id'].astype(str)
    # 按用户分组 → 把每个用户点的文章拼成句子
    docs = click_df.groupby(['user_id'])['click_article_id'].apply(lambda x: list(x)).reset_index()
    docs = docs['click_article_id'].values.tolist()

    # 为了方便查看训练的进度，这里设定一个log信息
    logging.basicConfig(format='%(asctime)s:%(levelname)s:%(message)s', level=logging.INFO)

    # 用 Word2Vec 训练 → 得到文章向量
    # 这里的参数对训练得到的向量影响也很大,默认负采样为5
    w2v = Word2Vec(
    docs,
    vector_size=16,   # ✅ 改这里
    sg=1,
    window=5,
    seed=2020,
    workers=24,
    min_count=1,
    epochs=1          # ✅ iter → epochs
)
    
    # 保存成字典的形式
    item_w2v_emb_dict = {k: w2v.wv[k] for k in click_df['click_article_id']}
    pickle.dump(item_w2v_emb_dict, open(save_path + 'item_w2v_emb.pkl', 'wb'))
    
    return item_w2v_emb_dict

# 可以通过字典查询对应的item的Embedding
# 一次性加载所有需要的 Embedding（文章内容、W2V、YouTube 文章 / 用户向量），没有就自动训练，直接喂给特征工程
def get_embedding(save_path, all_click_df):
    # 判断文件是否存在 → 存在直接读，不存在就训练
    if os.path.exists(save_path + 'item_content_emb.pkl'):
        item_content_emb_dict = pickle.load(open(save_path + 'item_content_emb.pkl', 'rb'))
    else:
        print('item_content_emb.pkl 文件不存在...')
        
    # w2v Embedding是需要提前训练好的
    if os.path.exists(save_path + 'item_w2v_emb.pkl'):
        item_w2v_emb_dict = pickle.load(open(save_path + 'item_w2v_emb.pkl', 'rb'))
    else:
        item_w2v_emb_dict = trian_item_word2vec(all_click_df)
        
    if os.path.exists(r'item_youtube_emb.pkl'):
        item_youtube_emb_dict = pickle.load(open(save_path + 'item_youtube_emb.pkl', 'rb'))
    else:
        print('item_youtube_emb.pkl 文件不存在...')
    
    if os.path.exists(save_path + 'user_youtube_emb.pkl'):
        user_youtube_emb_dict = pickle.load(open(save_path + 'user_youtube_emb.pkl', 'rb'))
    else:
        print('user_youtube_emb.pkl 文件不存在...')
    # 返回 4 个向量字典
    return item_content_emb_dict, item_w2v_emb_dict, item_youtube_emb_dict, user_youtube_emb_dict

# 读取文章的原始信息（类别、发布时间、字数），以便做文章特征
def get_article_info_df():
    article_info_df = pd.read_csv(data_path + 'articles.csv')  # 读取文章 csv 文件
    article_info_df = reduce_mem(article_info_df)  # 压缩内存（提速）
    
    return article_info_df

In [6]:
# 这里offline的online的区别就是验证集是否为空
click_trn, click_val, click_tst, val_ans = get_trn_val_tst_data(data_path, offline=False)

click_trn_hist, click_trn_last = get_hist_and_last_click(click_trn)

if click_val is not None:
    click_val_hist, click_val_last = click_val, val_ans
else:
    click_val_hist, click_val_last = None, None
    
click_tst_hist = click_tst

-- Mem. usage decreased to 23.34 Mb (69.4% reduction),time spend:0.00 min


# 训练数据负采样 + 召回数据打标签（笔记版）

---

## 一、负采样（Negative Sampling）

### 1️⃣ 为什么要做负采样？

在召回阶段，我们得到的数据形式为：

> (用户, 候选文章, 召回得分)

但此时存在一个核心问题：👉 **正负样本极度不平衡**

* 正样本（label=1）：用户实际点击的文章（≈ 3%）
* 负样本（label=0）：被召回但未点击的文章（≈ 97%）

---

### 🎯 负采样的两个核心目标

**(1) 解决样本不平衡**

* 避免模型被大量负样本主导
* 提高模型对正样本的学习能力

**(2) 降低计算成本**

* 减少数据量
* 加快后续特征工程 & 模型训练速度

---

### 2️⃣ 负采样原则（⚠️重点）

✅ **只对负样本采样**

* 正样本必须全部保留（信息最关键）

---

✅ **保证数据覆盖性**

* 每个用户必须保留
* 每篇文章尽量保留
* ❗避免采样后“用户/物品消失”

---

✅ **控制采样粒度（推荐）**

* 按用户采样，而不是全局随机采样
* 示例策略：

  > 每个用户最多保留 5 个负样本

👉 好处：

* 避免活跃用户数据爆炸
* 保持用户行为分布稳定

---

✅ **同步更新召回列表**

* 采样后数据结构会改变
* 后续特征工程必须基于“采样后的数据”

---

## 二、数据打标签（Labeling）

### 1️⃣ 打标签的目的

将召回结果转化为监督学习数据：

> (用户, 候选文章) → label

---

### 2️⃣ 标签定义

| 数据集 | 含义    | label |
| --- | ----- | ----- |
| 训练集 | 点击过   | 1     |
| 训练集 | 未点击   | 0     |
| 验证集 | 点击过   | 1     |
| 验证集 | 未点击   | 0     |
| 测试集 | 无真实点击 | -1    |

---

### ⚠️ 为什么测试集用 -1？

* 保持数据格式统一（方便 pipeline）
* 避免写额外分支逻辑
* 明确“不可用于训练”

---

## 三、整体流程（Pipeline）

### 🔄 Step 1：格式转换

将召回结果（通常是 dict）转换为结构化表：

```text
(user_id, item_id, score)
```

👉 转为 DataFrame，方便后续处理

---

### 🔄 Step 2：打标签

通过用户真实点击数据进行匹配：

```text
if (user, item) 在点击日志中 → label = 1
else → label = 0
```

---

### 🔄 Step 3：负采样

仅对以下数据进行采样：

* ✅ 训练集
* ✅ 验证集
* ❌ 测试集（不采样）

---

采样策略示例：

```text
每个用户最多保留 K 个负样本（如 K=5）
```

---

### 🔄 Step 4：输出数据

最终得到三份数据：

* 训练集（train）
* 验证集（valid）
* 测试集（test）

格式统一为：

```text
(user_id, item_id, score, label)
```

👉 可直接进入：

* 特征工程
* 排序模型（如 LightGBM / DNN）

---

## 🧠 一句话总结（面试/复述用）

> 负采样是在召回阶段后，通过对负样本进行下采样来缓解样本不平衡和计算压力，同时保留全部正样本，并保证用户和物品的覆盖性；打标签则是将召回数据转化为监督学习格式，为排序模型训练提供输入。

---


* 👉 或者讲清楚**为什么“每用户采样”比“全局采样”好（这个其实很关键）**


In [7]:
# 把嵌套的召回字典，转换成(user, item, score)的表格数据
def recall_dict_2_df(recall_list_dict):
    df_row_list = [] # [user, item, score]
    for user, recall_list in tqdm(recall_list_dict.items()):
        for item, score in recall_list:
            df_row_list.append([user, item, score])
    
    col_names = ['user_id', 'sim_item', 'score']
    recall_list_df = pd.DataFrame(df_row_list, columns=col_names)
    
    return recall_list_df

# 负采样函数，对负样本下采样，缓解样本失衡，保留所有用户/文章
def neg_sample_recall_data(recall_items_df, sample_rate=0.001):
    pos_data = recall_items_df[recall_items_df['label'] == 1]
    neg_data = recall_items_df[recall_items_df['label'] == 0]
    
    print('pos_data_num:', len(pos_data), 'neg_data_num:', len(neg_data), 'pos/neg:', len(pos_data)/len(neg_data))
    
    # 分组采样函数
    def neg_sample_func(group_df):
        neg_num = len(group_df)
        sample_num = max(int(neg_num * sample_rate), 1) # 保证最少有一个
        sample_num = min(sample_num, 5) # 保证最多不超过5个，可以根据实际情况进行选择
        return group_df.sample(n=sample_num, replace=True)
    
    # 对用户进行负采样，保证所有用户都在采样后的数据中
    neg_data_user_sample = neg_data.groupby('user_id', group_keys=False).apply(neg_sample_func)
    # 对文章进行负采样，保证所有文章都在采样后的数据中
    neg_data_item_sample = neg_data.groupby('sim_item', group_keys=False).apply(neg_sample_func)
    
    # 将上述两种情况下的采样数据合并
    neg_data_new = pd.concat([neg_data_user_sample, neg_data_item_sample])
    # 由于上述两个操作是分开的，可能将两个相同的数据给重复选择了，所以需要对合并后的数据进行去重
    neg_data_new = neg_data_new.sort_values(['user_id', 'score']).drop_duplicates(['user_id', 'sim_item'], keep='last')
    # 将正样本数据合并
    data_new = pd.concat([pos_data, neg_data_new], ignore_index=True)
    
    return data_new

# 给召回数据打标签：生成label（1=正样本，0=负样本，-1=测试集）
def get_rank_label_df(recall_list_df, label_df, is_test=False):
    if is_test:
        recall_list_df['label'] = -1
        return recall_list_df
    
    label_df = label_df.rename(columns={'click_article_id': 'sim_item'})
    recall_list_df_ = recall_list_df.merge(label_df[['user_id', 'sim_item', 'click_timestamp']], 
                                           how='left', on=['user_id', 'sim_item'])
    recall_list_df_['label'] = recall_list_df_['click_timestamp'].apply(lambda x: 0.0 if np.isnan(x) else 1.0)
    del recall_list_df_['click_timestamp']
    
    return recall_list_df_

# 总控函数
# 分别处理训练、验证、测试集：打标签+负采样
def get_user_recall_item_label_df(click_trn_hist, click_val_hist, click_tst_hist,click_trn_last, click_val_last, recall_list_df):
    # 训练集：打标签 + 负采样
    trn_user_items_df = recall_list_df[recall_list_df['user_id'].isin(click_trn_hist['user_id'].unique())]
    trn_user_item_label_df = get_rank_label_df(trn_user_items_df, click_trn_last, is_test=False)
    trn_user_item_label_df = neg_sample_recall_data(trn_user_item_label_df)
    
    # 验证集：打标签 + 负采样
    if click_val is not None:
        val_user_items_df = recall_list_df[recall_list_df['user_id'].isin(click_val_hist['user_id'].unique())]
        val_user_item_label_df = get_rank_label_df(val_user_items_df, click_val_last, is_test=False)
        val_user_item_label_df = neg_sample_recall_data(val_user_item_label_df)
    else:
        val_user_item_label_df = None
        
    # 测试集：只打标签(-1)，不做负采样
    tst_user_items_df = recall_list_df[recall_list_df['user_id'].isin(click_tst_hist['user_id'].unique())]
    tst_user_item_label_df = get_rank_label_df(tst_user_items_df, None, is_test=True)
    
    return trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df

In [8]:
# 读取召回列表
recall_list_dict = get_recall_list(save_path, single_recall_model='i2i_itemcf') # 这里只选择了单路召回的结果，也可以选择多路召回结果
# 将召回数据转换成df
recall_list_df = recall_dict_2_df(recall_list_dict)

# 给训练验证数据打标签，并负采样（这一部分时间比较久）
trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df = get_user_recall_item_label_df(click_trn_hist, 
                                                                                                       click_val_hist, 
                                                                                                       click_tst_hist,
                                                                                                       click_trn_last, 
                                                                                                       click_val_last, 
                                                                                                       recall_list_df)
trn_user_item_label_df.label

100%|██████████| 250000/250000 [00:00<00:00, 327296.65it/s]


pos_data_num: 64193 neg_data_num: 1935807 pos/neg: 0.033160847129905


0         1.0
1         1.0
2         1.0
3         1.0
4         1.0
         ... 
291130    0.0
291131    0.0
291132    0.0
291133    0.0
291134    0.0
Name: label, Length: 291135, dtype: float64

In [9]:
# 读取召回列表
recall_list_dict = get_recall_list(save_path, single_recall_model='i2i_itemcf') 
# 转换格式
recall_list_df = recall_dict_2_df(recall_list_dict)


100%|██████████| 250000/250000 [00:01<00:00, 169938.67it/s]


In [10]:
# recall里的用户
recall_users = set(recall_list_df['user_id'].unique())

# test里的用户
test_users = set(click_tst_hist['user_id'].unique())

print("recall用户数:", len(recall_users))
print("test用户数:", len(test_users))
print("交集用户数:", len(recall_users & test_users))

recall用户数: 250000
test用户数: 50000
交集用户数: 50000


In [11]:
trn_user_item_label_df.label

0         1.0
1         1.0
2         1.0
3         1.0
4         1.0
         ... 
291130    0.0
291131    0.0
291132    0.0
291133    0.0
291134    0.0
Name: label, Length: 291135, dtype: float64

In [12]:
# 将最终的召回的df数据转换成字典的形式做排序特征 转换成 {user: [(item, score, label), ...]} 的字典
def df_to_user_dict(df):
    return {
        user: list(zip(g['sim_item'], g['score'], g['label']))
        for user, g in df.groupby('user_id')
    }

# train
trn_user_item_label_tuples_dict = df_to_user_dict(trn_user_item_label_df)

# test
tst_user_item_label_tuples_dict = df_to_user_dict(tst_user_item_label_df)

# val
if val_user_item_label_df is not None:
    val_user_item_label_tuples_dict = df_to_user_dict(val_user_item_label_df)
else:
    val_user_item_label_tuples_dict = None

In [13]:
print("tst_user_item_label_df shape:", tst_user_item_label_df.shape)

tst_user_item_label_df shape: (500000, 4)


# 用户历史行为相关特征

---

## 一、核心理解

### 1. 这一步是干啥的？

这是整个特征工程的**核心环节（最重要）**。

前面的步骤：
- 数据集划分
- Embedding 训练
- 负采样

👉 本质都是在**准备数据**

而这一步：

> ✅ 把「用户点击行为」转成模型能用的特征

---

### 2. 为什么要做这个特征？

推荐系统核心逻辑：

> 用户的兴趣 = 历史点击行为

关键假设：

- 用户最近点击的文章 → 最能代表当前兴趣
- 候选文章是否被点击 → 取决于是否“像”用户之前点过的文章

---

### 3. 核心思路（一句话）

> 用“候选文章”和“用户历史文章”做对比 → 提取特征 → 给模型判断

---

## 二、具体做了什么（步骤拆解）

取每个用户最后 N 次点击的文章（N 默认 = 1，时效性优先，最近行为最有用）      
对用户的每一篇召回候选文章，分别和这 N 篇历史文章计算 3 类核心特征：    
相似度：两篇文章像不像（用 Embedding 向量点积）   
时间差：两篇文章的发布时间差（看用户喜不喜欢新文章）     
字数差：两篇文章的字数差（看用户喜欢长文 / 短文）       
计算相似度统计特征：最大、最小、总和、均值（让特征更稳定）    
补充辅助特征：召回得分、候选排名、标签          
把所有特征拼成一张大表，直接用于模型训练

In [14]:
click_tst.head()

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,249999,160974,1506959142820,4,1,17,1,13,2
1,249999,160417,1506959172820,4,1,17,1,13,2
2,249998,160974,1506959056066,4,1,12,1,13,2
3,249998,202557,1506959086066,4,1,12,1,13,2
4,249997,183665,1506959088613,4,1,17,1,15,5


In [15]:
# 读取文章特征
articles =  pd.read_csv(data_path+'articles.csv')
articles = reduce_mem(articles)

# 日志数据，就是前面的所有数据
if click_val is not None:
    all_data = pd.concat([click_trn, click_val, click_tst])
else:
    all_data = pd.concat([click_trn, click_tst])
    
all_data = reduce_mem(all_data)

# 拼上文章信息
all_data = all_data.merge(articles, left_on='click_article_id', right_on='article_id')

all_data.shape

-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min
-- Mem. usage decreased to 46.65 Mb (62.5% reduction),time spend:0.00 min


(1630633, 13)

In [16]:
# 下面基于data做历史相关的特征
def create_feature(users_id, recall_list, click_hist_df,  articles_info, articles_emb, user_emb=None, N=1):
    """
    基于用户的历史行为做相关特征
    :param users_id: 用户id
    :param recall_list: 对于每个用户召回的候选文章列表
    :param click_hist_df: 用户的历史点击信息
    :param articles_info: 文章信息
    :param articles_emb: 文章的embedding向量, 这个可以用item_content_emb, item_w2v_emb, item_youtube_emb
    :param user_emb: 用户的embedding向量， 这个是user_youtube_emb, 如果没有也可以不用， 但要注意如果要用的话， articles_emb就要用item_youtube_emb的形式， 这样维度才一样
    :param N: 最近的N次点击  由于testA日志里面很多用户只存在一次历史点击， 所以为了不产生空值，默认是1
    """
    
    # 建立一个二维列表保存结果， 后面要转成DataFrame
    all_user_feas = []
    i = 0
    for user_id in tqdm(users_id):
        # 该用户的最后N次点击
        hist_user_items = click_hist_df[click_hist_df['user_id']==user_id]['click_article_id'][-N:]
        
        # 遍历该用户的召回列表
        for rank, (article_id, score, label) in enumerate(recall_list[user_id]):
            # 该文章建立时间, 字数
            a_create_time = articles_info[articles_info['article_id']==article_id]['created_at_ts'].values[0]
            a_words_count = articles_info[articles_info['article_id']==article_id]['words_count'].values[0]
            single_user_fea = [user_id, article_id]
            # 计算与最后点击的商品的相似度的和， 最大值和最小值， 均值
            sim_fea = []
            time_fea = []
            word_fea = []
            # 遍历用户的最后N次点击文章
            for hist_item in hist_user_items:
                b_create_time = articles_info[articles_info['article_id']==hist_item]['created_at_ts'].values[0]
                b_words_count = articles_info[articles_info['article_id']==hist_item]['words_count'].values[0]
                
                sim_fea.append(np.dot(articles_emb[hist_item], articles_emb[article_id]))
                time_fea.append(abs(a_create_time-b_create_time))
                word_fea.append(abs(a_words_count-b_words_count))
                
            single_user_fea.extend(sim_fea)      # 相似性特征
            single_user_fea.extend(time_fea)    # 时间差特征
            single_user_fea.extend(word_fea)    # 字数差特征
            single_user_fea.extend([max(sim_fea), min(sim_fea), sum(sim_fea), sum(sim_fea) / len(sim_fea)])  # 相似性的统计特征
            
            if user_emb:  # 如果用户向量有的话， 这里计算该召回文章与用户的相似性特征 
                single_user_fea.append(np.dot(user_emb[user_id], articles_emb[article_id]))
                
            single_user_fea.extend([score, rank, label])    
            # 加入到总的表中
            all_user_feas.append(single_user_fea)
    
    # 定义列名
    id_cols = ['user_id', 'click_article_id']
    sim_cols = ['sim' + str(i) for i in range(N)]
    time_cols = ['time_diff' + str(i) for i in range(N)]
    word_cols = ['word_diff' + str(i) for i in range(N)]
    sat_cols = ['sim_max', 'sim_min', 'sim_sum', 'sim_mean']
    user_item_sim_cols = ['user_item_sim'] if user_emb else []
    user_score_rank_label = ['score', 'rank', 'label']
    cols = id_cols + sim_cols + time_cols + word_cols + sat_cols + user_item_sim_cols + user_score_rank_label
            
    # 转成DataFrame
    df = pd.DataFrame( all_user_feas, columns=cols)
    
    return df

article_info_df = get_article_info_df()
all_click = pd.concat([click_trn, click_tst])
item_content_emb_dict, item_w2v_emb_dict, item_youtube_emb_dict, user_youtube_emb_dict = get_embedding(save_path, all_click)



-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min


In [17]:
trn_user_item_feats_df = create_feature(trn_user_item_label_tuples_dict.keys(), trn_user_item_label_tuples_dict, 
                                            click_trn_hist, article_info_df, item_content_emb_dict)

if val_user_item_label_tuples_dict is not None:
    val_user_item_feats_df = create_feature(val_user_item_label_tuples_dict.keys(), val_user_item_label_tuples_dict, \
                                                click_val_hist, article_info_df, item_content_emb_dict)
else:
    val_user_item_feats_df = None
    
tst_user_item_feats_df = create_feature(tst_user_item_label_tuples_dict.keys(), tst_user_item_label_tuples_dict, \
                                            click_tst_hist, article_info_df, item_content_emb_dict)

100%|██████████| 50000/50000 [05:57<00:00, 139.68it/s]


In [18]:
# 1. 查看测试集 召回+标签字典 的长度（核心！）
print("测试集用户数量：", len(tst_user_item_label_tuples_dict.keys()))

# 2. 查看测试集历史点击数据
print("测试集历史点击数据行数：", click_tst_hist.shape)

# 3. 查看测试集原始标签数据
print("测试集标签字典内容（前2个）：", list(tst_user_item_label_tuples_dict.items())[:2])

测试集用户数量： 50000
测试集历史点击数据行数： (518010, 9)
测试集标签字典内容（前2个）： [(200000, [(194935, 1.9285190204939584, -1), (237870, 1.7741596518635225, -1), (195645, 1.0686706646572088, -1), (50573, 1.0316675868441805, -1), (16132, 0.93396175580685, -1), (63344, 0.9190202394146835, -1), (187005, 0.9101811770766249, -1), (255153, 0.8562233448960022, -1), (195773, 0.8139436028246723, -1), (194421, 0.8129426721039774, -1)]), (200001, [(64329, 1.3296669537990542, -1), (272143, 1.0484939914130704, -1), (324823, 0.9868849362193984, -1), (199198, 0.8993189091846294, -1), (166581, 0.7331954087316326, -1), (224658, 0.6088708996692251, -1), (285808, 0.5849067923399364, -1), (285343, 0.5393348689226106, -1), (198659, 0.44065596329888895, -1), (299697, 0.3365032312251979, -1)])]


In [19]:
# 保存一份省的每次都要重新跑，每次跑的时间都比较长
trn_user_item_feats_df.to_csv(save_path + 'trn_user_item_feats_df.csv', index=False)

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(save_path + 'val_user_item_feats_df.csv', index=False)

tst_user_item_feats_df.to_csv(save_path + 'tst_user_item_feats_df.csv', index=False)    

## 根据点击时间和点击文章的次数，区分 高频活跃用户 / 低频冷门用户
判断逻辑：    
点击文章次数越多 + 两次点击的时间间隔越短 = 越活跃       
点击次数少 + 间隔很长 = 不活跃    

计算步骤：
   ① 按user_id分组，统计每个用户的点击文章总次数、两两点击的时间间隔均值     
   ② 特殊处理：仅点击1次的用户，时间间隔均值直接赋值1       
   ③ 点击次数取倒数，将点击次数、时间间隔均值分别归一化到0-1区间        
   ④ 两者相加得到active_level，该值**越小代表用户越活跃**


In [20]:
def active_level(all_data, cols):
    """
    核心作用：输入用户点击日志，输出每个用户的活跃度分数
    输入：全量点击数据、需要用的列（用户ID/文章ID/点击时间）
    输出：用户ID + 活跃度分数（active_level）的表格
    """
    data = all_data[cols]
    # 按用户+点击时间排序，保证时间顺序正确
    data.sort_values(['user_id', 'click_timestamp'], inplace=True)
    
    # 分组统计：每个用户 → 点击总次数(click_size) + 点击时间列表(click_timestamp)
    user_act = pd.DataFrame(data.groupby('user_id', as_index=False)[['click_article_id', 'click_timestamp']].
                            agg({'click_article_id':np.size, 'click_timestamp': {list}}).values, columns=['user_id', 'click_size', 'click_timestamp'])
    
    # 计算：两次点击的平均时间间隔
    def time_diff_mean(l):
        # 如果只点击1次，无间隔，返回1
        if len(l) == 1:
            return 1
        # 多次点击：计算相邻时间差的均值
        else:
            return np.mean([j-i for i, j in list(zip(l[:-1], l[1:]))])
        
    user_act['time_diff_mean'] = user_act['click_timestamp'].apply(lambda x: time_diff_mean(x))
    
    # 点击次数取倒数（次数越多，数值越小）
    user_act['click_size'] = 1 / user_act['click_size']
    
    # 两个指标归一化（缩放到0~1，统一量纲）
    user_act['click_size'] = (user_act['click_size'] - user_act['click_size'].min()) / (user_act['click_size'].max() - user_act['click_size'].min())
    user_act['time_diff_mean'] = (user_act['time_diff_mean'] - user_act['time_diff_mean'].min()) / (user_act['time_diff_mean'].max() - user_act['time_diff_mean'].min())     
    
    # 合并最终活跃度特征：值越小 → 越活跃
    user_act['active_level'] = user_act['click_size'] + user_act['time_diff_mean']
    
    # 清理无用列，返回结果
    del user_act['click_timestamp']
    return user_act

# 调用函数，生成用户活跃度特征表
user_act_fea = active_level(all_data, ['user_id', 'click_article_id', 'click_timestamp'])

In [21]:
user_act_fea.head()    # 第三列有用

,user_id,click_size,time_diff_mean,active_level
0,0,0.499466,0.000048,0.499515
1,1,0.499466,0.000048,0.499515
2,2,0.499466,0.000048,0.499515
3,3,0.499466,0.000048,0.499515
4,4,0.499466,0.000048,0.499515


## 根据点击时间 被点击次数 区分 热门文章/冷门文章
判定逻辑：    
   - 文章被点击次数越多 + 点击时间间隔越短 → 越热门       
计算步骤：                
   ① 按 click_article_id 分组，统计每篇文章的点击用户数、两两点击的时间间隔均值         
   ② 特殊处理：仅被点击1次的文章，时间间隔均值直接赋值1          
   ③ 点击用户数取倒数，将时间间隔均值分别归一化到0-1区间      
   ④ 两者相加得到 hot_level ，该值**越小代表文章越热门**（被点击的次数越大且时间间隔越短）           


In [22]:
def hot_level(all_data, cols):
    """
    制作衡量文章热度的特征
    :param all_data: 完整的点击数据
    :param cols: 需要用到的列：['user_id', 'click_article_id', 'click_timestamp']
    """
    # 只取需要的列：用户ID、文章ID、点击时间
    data = all_data[cols]
    
    # 按【文章ID + 点击时间】排序，保证时间顺序正确
    # 必须排序！否则时间间隔会算错
    data.sort_values(['click_article_id', 'click_timestamp'], inplace=True)

    # 按文章分组统计 
    # 对每一篇文章，统计两件事：
    # 1) 被多少用户点击 → user_num
    # 2) 所有点击时间 → 存成一个列表
    article_hot = pd.DataFrame(
        data.groupby('click_article_id', as_index=False)[['user_id', 'click_timestamp']].
        agg({
            'user_id': np.size,        # 统计点击次数（多少人点了这篇文章）
            'click_timestamp': list    # 把点击时间变成一个列表 [t1, t2, t3...]
        }).values, 
        columns=['click_article_id', 'user_num', 'click_timestamp']
    )

    # 计算平均点击时间间隔 
    # 定义函数：计算时间列表的平均间隔
    def time_diff_mean(l):
        # 如果文章只被点击1次 → 没有间隔 → 返回1
        if len(l) == 1:
            return 1
        # 否则：计算相邻两次点击的时间差 → 求平均值
        else:
            return np.mean([j - i for i, j in list(zip(l[:-1], l[1:]))])
    
    # 对每篇文章应用函数，得到平均时间间隔
    article_hot['time_diff_mean'] = article_hot['click_timestamp'].apply(lambda x: time_diff_mean(x))

    # 点击次数取倒数 
    # 点击次数越多 → 倒数越小 → 热度分数越低（越热门）
    article_hot['user_num'] = 1 / article_hot['user_num']

    # 两个特征归一化到 0~1 
    # 归一化目的：让点击数和时间间隔在同一个尺度上相加
    
    # 归一化点击数
    article_hot['user_num'] = (article_hot['user_num'] - article_hot['user_num'].min()) / \
                             (article_hot['user_num'].max() - article_hot['user_num'].min())
    
    # 归一化平均时间间隔
    article_hot['time_diff_mean'] = (article_hot['time_diff_mean'] - article_hot['time_diff_mean'].min()) / \
                                    (article_hot['time_diff_mean'].max() - article_hot['time_diff_mean'].min())

    #  合并成最终热度特征 
    # 两个值相加：数值越小 = 点击越多 + 间隔越短 = 越热门
    article_hot['hot_level'] = article_hot['user_num'] + article_hot['time_diff_mean']

    #  格式整理 
    article_hot['click_article_id'] = article_hot['click_article_id'].astype('int')
    del article_hot['click_timestamp']  # 删除中间列

    return article_hot

# 调用函数，生成文章热度特征
article_hot_fea = hot_level(all_data, ['user_id', 'click_article_id', 'click_timestamp'])    

In [23]:
article_hot_fea.head()   # 最后一列有用

,click_article_id,user_num,time_diff_mean,hot_level
0,3,1.0,0.0,1.0
1,69,1.0,0.0,1.0
2,84,1.0,0.0,1.0
3,94,1.0,0.0,1.0
4,125,1.0,0.0,1.0


## 用户的系列习惯    
用户的设备习惯， 这里取最常用的设备（众数）    

用户的时间习惯： 根据其点击过得历史文章的时间来做一个统计（这个感觉最好是把时间戳里的时间特征的h特征提出来，看看用户习惯一天的啥时候点击文章）， 但这里先用转换的时间吧， 求个均值        

用户的爱好特征， 对于用户点击的历史文章主题进行用户的爱好判别， 更偏向于哪几个主题， 这个最好是multi-hot进行编码， 先试试行不    
       
用户文章的字数差特征， 用户的爱好文章的字数习惯

### 1.用户设备习惯   
1. 特征目的：提取用户固定的设备使用习惯，属于用户**静态画像特征**      
2. 核心逻辑：   
   一个用户可能用多个设备，取**出现次数最多的设备（众数）** 代表用户的常用设备    
3. 统计方式：          
   ① 按 user_id 分组       
   ② 对所有设备相关列，批量计算众数        
   ③ 生成每个用户唯一的设备特征表       
4. 特征列：设备环境、设备组、操作系统、国家、地区、来源类型

In [24]:
def device_fea(all_data, cols):
    """
    制作用户的设备特征
    :param all_data: 全量用户点击日志数据
    :param cols: 需要计算的设备特征列
    """
    # 1. 筛选出用户ID + 所有设备相关的列，只保留需要的数据
    user_device_info = all_data[cols]
    
    # 2. 按用户ID分组，对每一列计算【众数】（出现次数最多的值）
    # x.value_counts().index[0]：统计列值频次，取第一个（众数）
    user_device_info = user_device_info.groupby('user_id').agg(lambda x: x.value_counts().index[0]).reset_index()
    
    # 3. 返回每个用户对应的常用设备特征表
    return user_device_info

# 定义需要提取的设备特征列（所有和用户设备/环境相关的字段）
device_cols = [
    'user_id', 
    'click_environment',   # 点击环境
    'click_deviceGroup',   # 设备组别（手机/电脑等）
    'click_os',           # 操作系统
    'click_country',      # 国家
    'click_region',       # 地区
    'click_referrer_type' # 来源类型
]
# 调用函数，生成用户设备特征
user_device_info = device_fea(all_data, device_cols)
user_device_info.head()

,user_id,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,4,1,17,1,25,2
1,1,4,1,17,1,25,6
2,2,4,3,20,1,25,2
3,3,4,3,2,1,25,2
4,4,4,1,12,1,16,1


### 2.用户时间习惯     
1. 特征目的：提取用户的**时间偏好**，刻画用户喜欢在什么时间点击、喜欢什么时间发布的文章          
2. 核心逻辑：        
   ① 按 user_id 分组                     
   ② 对时间戳做**归一化**（消除数值过大问题）         
   ③ 计算均值，代表用户的平均点击时间、偏好的文章发布时间          
3. 两个时间特征：          
   - user_time_hob1：用户平均**点击时间**偏好         
   - user_time_hob2：用户偏好的**文章发布时间**

In [25]:
def user_time_hob_fea(all_data, cols):
    """
    制作用户的时间习惯特征
    :param all_data: 全量用户点击日志数据
    :param cols: 需要计算的时间特征列
    """
    # 1. 筛选出用户ID + 点击时间戳 + 文章发布时间戳
    user_time_hob_info = all_data[cols]
    
    # 2. 时间戳归一化（MinMaxScaler把数值缩放到 0~1 之间）
    # 目的：时间戳数值极大，归一化后方便模型训练
    mm = MinMaxScaler()
    user_time_hob_info['click_timestamp'] = mm.fit_transform(user_time_hob_info[['click_timestamp']])
    user_time_hob_info['created_at_ts'] = mm.fit_transform(user_time_hob_info[['created_at_ts']])

    # 3. 按用户ID分组，对两个时间特征计算**均值**
    user_time_hob_info = user_time_hob_info.groupby('user_id').agg('mean').reset_index()
    
    # 4. 重命名列名，方便识别特征含义
    user_time_hob_info.rename(
        columns={
            'click_timestamp': 'user_time_hob1', 
            'created_at_ts': 'user_time_hob2'
        }, 
        inplace=True
    )
    
    # 5. 返回用户时间偏好特征表
    return user_time_hob_info

# 定义需要提取的时间特征列
user_time_hob_cols = ['user_id', 'click_timestamp', 'created_at_ts']
# 调用函数，生成用户时间习惯特征
user_time_hob_info = user_time_hob_fea(all_data, user_time_hob_cols)
user_time_hob_info.head()

,user_id,user_time_hob1,user_time_hob2
0,0,0.343715,0.992865
1,1,0.343618,0.992721
2,2,0.343651,0.992020
3,3,0.343629,0.992774
4,4,0.343702,0.992688


### 3.用户主题爱好特征    
1. 特征目的：提取用户喜欢的文章主题类别，是推荐系统**最核心的用户偏好特征**      
2. 核心逻辑：      
   ① 按 user_id 分组         
   ② 把用户历史点击的所有文章类别 `category_id` 聚合成一个**列表**     
3. 后续用途：       
   制作交叉特征 → 候选文章主题在列表中=1（匹配喜好），否则=0     
4. 特点：不直接做数值，只存偏好列表，为后续特征做准备

In [26]:
# 把用户所有点击过的文章的主题 ID（category_id）全部收集起来，做成一个列表，直接作为用户的主题爱好特征
def user_cat_hob_fea(all_data, cols):
    """
    用户的主题爱好特征
    :param all_data: 全量用户点击日志数据
    :param cols: 需要的列：[用户ID, 文章类别ID]
    """
    # 1. 筛选列：只保留用户ID和文章类别ID
    user_category_hob_info = all_data[cols]
    
    # 2. 按用户ID分组，将每个用户点击的所有文章类别聚合成【列表】
    user_category_hob_info = user_category_hob_info.groupby('user_id').agg({list}).reset_index()
    
    # 3. 新建干净的DataFrame，只保留用户ID和类别列表
    user_cat_hob_info = pd.DataFrame()
    user_cat_hob_info['user_id'] = user_category_hob_info['user_id']
    # 存储用户所有历史点击的文章类别，列名cate_list
    user_cat_hob_info['cate_list'] = user_category_hob_info['category_id']
    
    # 4. 返回用户主题偏好列表
    return user_cat_hob_info

# 定义特征列：用户ID + 文章类别ID
user_category_hob_cols = ['user_id', 'category_id']
# 调用函数生成特征
user_cat_hob_info = user_cat_hob_fea(all_data, user_category_hob_cols)
user_cat_hob_info.head()

,user_id,cate_list
0,0,"[26, 281]"
1,1,"[418, 133]"
2,2,"[43, 297]"
3,3,"[99, 43]"
4,4,"[67, 66]"


### 4. 用户数字偏好特征    
1. 特征目的：统计用户喜欢的文章字数，刻画用户阅读习惯（长文/短文）      
2. 核心逻辑：     
   ① 按 user_id 分组      
   ② 计算用户点击文章的**平均字数**         
3. 数值含义：平均值越大 → 越喜欢长文，越小 → 越喜欢短文      
4. 特点：极简统计特征，计算成本低、效果好

In [27]:
# 按用户ID分组，计算点击文章字数的平均值，重命名列
user_wcou_info = all_data.groupby('user_id')['words_count'].agg('mean').reset_index()
user_wcou_info.rename(columns={'words_count': 'words_hbo'}, inplace=True)
user_wcou_info.head()

,user_id,words_hbo
0,0,266.0
1,1,169.0
2,2,210.0
3,3,196.5
4,4,220.0


### 用户特征合并保存

In [28]:
# 1. 所有用户特征表按 user_id 合并
user_info = pd.merge(user_act_fea, user_device_info, on='user_id')
user_info = user_info.merge(user_time_hob_info, on='user_id')
user_info = user_info.merge(user_cat_hob_info, on='user_id')
user_info = user_info.merge(user_wcou_info, on='user_id')

# 2. 保存到文件，后续直接读取
user_info.to_csv(save_path + 'user_info.csv', index=False)

In [29]:
# 把用户信息直接读入进来
user_info = pd.read_csv(save_path + 'user_info.csv')
# 读取训练集历史行为特征
if os.path.exists(save_path + 'trn_user_item_feats_df.csv'):
    trn_user_item_feats_df = pd.read_csv(save_path + 'trn_user_item_feats_df.csv')
# 读取测试集历史行为特征 
if os.path.exists(save_path + 'tst_user_item_feats_df.csv'):
    tst_user_item_feats_df = pd.read_csv(save_path + 'tst_user_item_feats_df.csv')
# 读取验证集历史行为特征
if os.path.exists(save_path + 'val_user_item_feats_df.csv'):
    val_user_item_feats_df = pd.read_csv(save_path + 'val_user_item_feats_df.csv')
else:
    val_user_item_feats_df = None


In [30]:
# 拼上用户特征（核心步骤）

# 训练集：用户特征 + 历史行为特征 合并
trn_user_item_feats_df = trn_user_item_feats_df.merge(user_info, on='user_id', how='left')

# 验证集：用户特征 + 历史行为特征 合并
if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(user_info, on='user_id', how='left')
else:
    val_user_item_feats_df = None
    
# 测试集：用户特征 + 历史行为特征 合并
tst_user_item_feats_df = tst_user_item_feats_df.merge(user_info, on='user_id',how='left')

# 查看合并后的所有特征列名
trn_user_item_feats_df.columns

Index(['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0',
       'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'label',
       'click_size', 'time_diff_mean', 'active_level', 'click_environment',
       'click_deviceGroup', 'click_os', 'click_country', 'click_region',
       'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'cate_list',
       'words_hbo'],
      dtype='object')

### 文章特征读取拼接   
#### 核心思路（可直接复制）     
1. 操作目的：读取文章原始基础信息，将**文章侧特征**拼接至总特征表        
2. 拼接依据：          
   特征表的`click_article_id`（候选文章ID） ↔ 文章表的`article_id`          
3. 拼接完成后：          
   总特征表 = 用户历史行为特征 + 用户全量画像特征 + 文章基础特征       
4. 这是**特征工程的最后一步拼接**，三大特征全部融合       

In [31]:
# 读取文章原始基础信息表（类别、发布时间、字数等）
articles =  pd.read_csv(data_path+'articles.csv')
# 压缩内存，减少数据占用空间，提升运行速度
articles = reduce_mem(articles)

# 训练集：拼接文章特征（核心：文章ID匹配）
trn_user_item_feats_df = trn_user_item_feats_df.merge(
    articles, 
    left_on='click_article_id',   # 左侧特征表的文章ID列
    right_on='article_id'         # 右侧文章表的文章ID列
)

# 验证集：拼接文章特征
if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(
        articles, 
        left_on='click_article_id', 
        right_on='article_id'
    )
else:
    val_user_item_feats_df = None

# 测试集：拼接文章特征
tst_user_item_feats_df = tst_user_item_feats_df.merge(
    articles, 
    left_on='click_article_id', 
    right_on='article_id'
)

-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min


## 主题匹配特征 + 特征清理 + 最终保存       
1. 制作核心匹配特征：is_cat_hab      
   判断召回文章的主题，是否在用户的历史爱好列表中 → 匹配=1，不匹配=0       
2. 删除无用特征：       
   - cate_list：列表格式，模型无法训练，用完即删      
   - article_id：冗余ID列，无建模价值       
3. 保存最终特征表：      
   生成可直接用于模型训练的完整版特征文件，特征工程全部结束

In [32]:
print(trn_user_item_feats_df.columns)

Index(['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0',
       'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'label',
       'click_size', 'time_diff_mean', 'active_level', 'click_environment',
       'click_deviceGroup', 'click_os', 'click_country', 'click_region',
       'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'cate_list',
       'words_hbo', 'article_id', 'category_id', 'created_at_ts',
       'words_count'],
      dtype='object')


In [33]:
print(tst_user_item_feats_df.shape)

(500000, 29)


In [34]:
print("tst size:", len(tst_user_item_feats_df))

tst size: 500000


In [35]:
print(type(tst_user_item_feats_df['cate_list'].iloc[0]))

<class 'str'>


In [36]:
# 1. 【核心】制作主题匹配特征：文章主题是否在用户爱好中
trn_user_item_feats_df['is_cat_hab'] = trn_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)
if val_user_item_feats_df is not None:
    val_user_item_feats_df['is_cat_hab'] = val_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)
tst_user_item_feats_df['is_cat_hab'] = tst_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)

# 2. 删除无用特征（模型不能识别列表/冗余ID）
del trn_user_item_feats_df['cate_list']
del trn_user_item_feats_df['article_id']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['cate_list']
    del val_user_item_feats_df['article_id']

del tst_user_item_feats_df['cate_list']
del tst_user_item_feats_df['article_id']

# 3. 保存最终完整版特征（直接用于模型训练）
trn_user_item_feats_df.to_csv(save_path + 'trn_user_item_feats_df.csv', index=False)
if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(save_path + 'val_user_item_feats_df.csv', index=False)
tst_user_item_feats_df.to_csv(save_path + 'tst_user_item_feats_df.csv', index=False)